# Варіант: [Впишіть ваше місто/варіант]
Виконання завдань 1-4.

In [ ]:
import pandas as pd

months = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]

# УВАГА: Підставте дані свого варіанта замість цих чисел!
avg_temp_data = [-3.5, -2.0, 3.3, 10.2, 16.1, 19.5, 21.3, 20.4, 14.5, 8.2, 2.5, -2.0]
precip_data = [36, 39, 39, 46, 57, 82, 71, 60, 57, 41, 50, 47]

df = pd.DataFrame({
    "month": months,
    "avg_temp": avg_temp_data,
    "precip": precip_data,
})

print("--- Перші 5 рядків ---")
display(df.head())

print("\n--- Форма DataFrame (рядки, стовпці) ---")
print(df.shape)

print("\n--- Типи даних стовпців ---")
print(df.dtypes)

### Завдання 1. Індекс

In [ ]:
# 1. Виведення індексу за замовчуванням
print("Індекс за замовчуванням:", df.index)

# 2. Заміна індексу на стовпець month
df_by_month = df.set_index("month")
print("\nРядок для Травня через .loc:")
display(df_by_month.loc["Тра"]) # або будь-який інший місяць

# 3. Повернення звичайної нумерації (повертаємо у вихідний df)
df_by_month.reset_index(inplace=True)
print("\nПісля скидання індексу:")
display(df_by_month.head(2))

### Завдання 2. Відбір рядків і стовпців

In [ ]:
# Встановлюємо числовий індекс від 1 до 12
df.index = range(1, 13)

# 1. Відбір через .loc (за підписом)
print("--- Відбір через .loc (підписи з 1 по 6 включно) ---")
display(df.loc[1:6, "avg_temp"])

# 2. Відбір через .iloc (за позицією)
# avg_temp - це другий стовпець, його позиція 1 (0 - month, 1 - avg_temp, 2 - precip)
print("\n--- Відбір через .iloc (позиції з 0 по 5) ---")
display(df.iloc[0:6, 1])

# 3. Лише стовпці month і precip для всіх рядків
print("\n--- Стовпці month і precip ---")
display(df[["month", "precip"]])

**Пояснення до .loc та .iloc:**
- `.loc[1:6]` шукає рядки, де **ім'я (підпис) індексу** дорівнює від 1 до 6 включно (права межа включається).
- `.iloc[0:6]` бере рядки за їх **фізичним порядком (позицією)** у пам'яті, починаючи з 0 (права межа *не включається*, тобто бере позиції 0, 1, 2, 3, 4, 5).
Оскільки наші підписи (1-6) збігаються з першими шістьма позиціями (0-5), результат однаковий, але логіка роботи методів різна.

### Завдання 3. Фільтрація за умовою

In [ ]:
print("--- Місяці, де avg_temp > 15 ---")
display(df[df["avg_temp"] > 15])

print("\n--- Місяці, де avg_temp > 10 І precip > 50 ---")
display(df[(df["avg_temp"] > 10) & (df["precip"] > 50)])

print("\n--- Місяці, де avg_temp < 0 АБО precip > 80 ---")
display(df[(df["avg_temp"] < 0) | (df["precip"] > 80)])

### Завдання 4. Обчислювані стовпці

In [ ]:
# 1. Температура у Фаренгейтах (векторизована формула)
df["avg_temp_f"] = df["avg_temp"] * 9/5 + 32

# 2. Булевий стовпець is_wet
avg_precip = df["precip"].mean()
df["is_wet"] = df["precip"] > avg_precip

# 3. Текстовий стовпець season через .apply()
def get_season(month_num):
    if month_num in [12, 1, 2]:
        return "зима"
    elif month_num in [3, 4, 5]:
        return "весна"
    elif month_num in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"

# Оскільки наш індекс наразі це номер місяця (1-12), створимо з нього стовпець для apply
df["month_num"] = df.index
df["season"] = df["month_num"].apply(get_season)

# Видалимо допоміжний стовпець, щоб таблиця виглядала охайно
df.drop(columns=["month_num"], inplace=True)

display(df)

### Відповіді на питання:

**1. Чи можете пояснити, чому .loc[1:6] і .iloc[0:6] можуть повернути різну кількість рядків, залежно від того, який саме індекс у таблиці?**
Метод `.loc` працює з **мітками (назвами) індексів** і включає обидва кінці зрізу (включно з 6). Якщо індекси йдуть не по порядку (наприклад, `[1, 5, 2, 6]`), `.loc[1:6]` поверне всі рядки, що знаходяться між міткою `1` і міткою `6`. 
Метод `.iloc` працює строго з **позиціями рядків** (від 0 і далі), і його зріз працює як стандартний зріз у Python: права межа *не включається*. Тому `.iloc[0:6]` завжди поверне рівно 6 рядків (позиції 0, 1, 2, 3, 4, 5) незалежно від того, як названі індекси.

**2. Чи розумієте, чому df["avg_temp"] > 10 and df["precip"] > 50 викличе помилку, а (df["avg_temp"] > 10) & (df["precip"] > 50) — ні?**
Стандартний оператор Python `and` намагається оцінити істинність *всього об'єкта Series (стовпця)* загалом, що неможливо зробити однозначно (виникає помилка "The truth value of a Series is ambiguous"). Оператор `&` виконує **поелементне (побітове) логічне множення** для кожного рядка таблиці окремо. Дужки обов'язкові, оскільки оператор `&` має вищий пріоритет, ніж оператори порівняння (`>`).

**3. Чи можете описати словами, у чому різниця між обчислюваним стовпцем, створеним формулою, і створеним через .apply()?**
- **Формула (векторизація):** Застосовується відразу до всього стовпця (масиву даних). Вона використовує оптимізовані бібліотеки на мові C під капотом pandas/numpy, тому працює надзвичайно швидко. Підходить для базових математичних операцій.
- **.apply():** Працює як цикл (iterating). Він бере власну функцію Python і застосовує її рядок за рядком до кожного елемента окремо. Це працює повільніше, але дозволяє використовувати складну нетривіальну логіку (як-от `if/elif/else`, обробку тексту тощо), яку важко або неможливо записати однією простою формулою.